
# NOVA CORPORATION | BR001
## DATA QUALITY — INVESTIGATION WORKBOOK

**Business Request:** BR001 — Executive Performance Review  
**Company:** Nova Corporation (synthetic corporate analytics simulation)  
**Reporting Period:** H1 2025 vs. H1 2026  
**Source:** BR001 analytical extract  
**Technology:** Python / pandas  
**Master DQ Register:** `Nova_BR001_Data_Quality_Workbook.xlsx`  
**Execution Policy:** Read-only analysis; no source remediation  
**Evidence Policy:** Record findings, counts, and decisions in the master workbook; preserve technical evidence here.

### PURPOSE

Investigate and quantify outstanding data-quality findings, retain reproducible evidence, and support subsequent business clarification.

### METHODOLOGY

01. Environment & Data Loading
02. Investigation Register
03. Automated Data Quality Assessment
04. Order Identity & Consistency
05. Transaction Status & Financial Exposure
06. Statistical Outlier Reconciliation
07. DQ013 Investigation Findings
08. Duplicate Sales Line Investigation
09. Invalid Numeric Values

### GOVERNANCE

- The source warehouse remains unchanged.
- No record is removed, corrected, or reclassified solely on the basis of this notebook.
- Technical findings must be reconciled with the master DQ workbook.
- Clarification decisions must be documented before remediation.

---

## 00 | Environment & Data Loading

**Objective:** Load the analytical extract and reconcile it against the validated SQL extraction baseline.


In [1]:

from pathlib import Path
import pandas as pd

# ============================================================
# BR001 | PROJECT ROOT RESOLUTION
# ============================================================

def find_project_root(start_path=None):
    start = Path(start_path or Path.cwd()).resolve()

    for candidate in (start, *start.parents):
        required_paths = [
            candidate / "03_sql",
            candidate / "04_data",
            candidate / "05_analysis",
        ]

        if all(path.is_dir() for path in required_paths):
            return candidate

    raise FileNotFoundError(
        "Nova_Corporation project root was not found. "
        "Open VS Code from the project folder and retry."
    )

PROJECT_ROOT = find_project_root()

print("Project root:", PROJECT_ROOT)
print("Project root validation: PASS")

source_path = (
    PROJECT_ROOT
    / "04_data"
    / "raw"
    / "br001_raw_analytical_extract.csv"
)

# Load source data without modifying the original file
df = pd.read_csv(
    source_path,
    dtype={
        "sales_line_id": "string",
        "order_id": "string",
        "customer_id": "string",
        "channel_id": "string"
    }
)

# Reconcile against the validated SQL baseline
baseline = {
    "Rows": 23530,
    "Gross Revenue": 17683230.67,
    "Discount Amount": 2745693.85,
    "Net Revenue": 14937536.82,
    "Total Cost": 10442161.65,
    "Gross Profit": 4495375.17
}

columns = {
    "Gross Revenue": "gross_revenue",
    "Discount Amount": "discount_amount",
    "Net Revenue": "net_revenue",
    "Total Cost": "total_cost",
    "Gross Profit": "gross_profit"
}

results = []

for metric, expected in baseline.items():
    actual = (
        len(df)
        if metric == "Rows"
        else round(df[columns[metric]].sum(), 2)
    )

    difference = round(actual - expected, 2)

    results.append({
        "Metric": metric,
        "SQL Baseline": expected,
        "Python Result": actual,
        "Difference": difference,
        "Status": "PASS" if abs(difference) < 0.01 else "FAIL"
    })

reconciliation = pd.DataFrame(results)

display(reconciliation)

assert reconciliation["Status"].eq("PASS").all(), (
    "Source reconciliation failed. Investigate before proceeding."
)


Project root: Nova_Corporation
Project root validation: PASS


,Metric,SQL Baseline,Python Result,Difference,Status
0,Rows,23530.00,23530.00,0.0,PASS
1,Gross Revenue,17683230.67,17683230.67,0.0,PASS
2,Discount Amount,2745693.85,2745693.85,0.0,PASS
3,Net Revenue,14937536.82,14937536.82,0.0,PASS
4,Total Cost,10442161.65,10442161.65,0.0,PASS
5,Gross Profit,4495375.17,4495375.17,0.0,PASS



---

## 01 | Investigation Register

**Objective:** Document outstanding clarification and data-quality investigations.

**Traceability:** CL-001 and other registered investigations.

**Master Record:** `Nova_BR001_Data_Quality_Workbook.xlsx`

### INVESTIGATION PRINCIPLE

This register provides an operational overview of unresolved investigations.

It does not replace the master DQ workbook or authorize remediation.

All findings, evidence references, and final decisions must remain traceable to the master record.


In [2]:

# ============================================================
# BR001 | INVESTIGATION REGISTER
# ============================================================

investigations = [
    {
        "Reference": "CL-001",
        "Investigation": "Order identity and consistency",
        "Focus": "Conflicting customers and dates within orders"
    },
    {
        "Reference": "DQ004 / DQ009",
        "Investigation": "Missing references",
        "Focus": "Missing customer and channel information"
    },
    {
        "Reference": "DQ005",
        "Investigation": "Duplicate sales line identifiers",
        "Focus": "Conflicting records and financial impact"
    },
    {
        "Reference": "DQ007",
        "Investigation": "Invalid numeric values",
        "Focus": "Business rule violations and impact"
    },
    {
        "Reference": "DQ013",
        "Investigation": "Statistical outliers",
        "Focus": "Anomaly classification and residual risk"
    },
    {
        "Reference": "DQ015 / CL-002",
        "Investigation": "Partially returned transactions",
        "Focus": "Financial treatment and KPI eligibility"
    },
    {
        "Reference": "CL-003",
        "Investigation": "Undocumented payment method",
        "Focus": "Source domain clarification"
    }
]

investigation_register = pd.DataFrame(investigations)

investigation_register["Status"] = "PENDING INVESTIGATION"

display(investigation_register)


,Reference,Investigation,Focus,Status
0,CL-001,Order identity and consistency,Conflicting customers and dates within orders,PENDING INVESTIGATION
1,DQ004 / DQ009,Missing references,Missing customer and channel information,PENDING INVESTIGATION
2,DQ005,Duplicate sales line identifiers,Conflicting records and financial impact,PENDING INVESTIGATION
3,DQ007,Invalid numeric values,Business rule violations and impact,PENDING INVESTIGATION
4,DQ013,Statistical outliers,Anomaly classification and residual risk,PENDING INVESTIGATION
5,DQ015 / CL-002,Partially returned transactions,Financial treatment and KPI eligibility,PENDING INVESTIGATION
6,CL-003,Undocumented payment method,Source domain clarification,PENDING INVESTIGATION




---

## 02 | Automated Data Quality Assessment

**Objective:** Investigate selected missing references and numeric business-rule violations within the BR001 analytical extract.

**Traceability:** DQ004 / DQ007 / DQ009  
**Source:** BR001 Analytical Extract  
**Master Record:** `Nova_BR001_Data_Quality_Workbook.xlsx`

### VALIDATION SCOPE

This notebook provides targeted, supplementary investigation of selected data-quality conditions. It does not replace the complete validation evidence maintained in the master Data Quality workbook.

**DQ004 — Null / Blank Completeness**

The master workbook contains the full-population completeness assessment covering all 36 fields across 23,530 sales lines.

The Python investigation focuses on selected missing references and their potential impact on downstream analysis.

**DQ007 — Range & Domain Validity**

The master workbook documents the approved numeric range and categorical domain assessments.

The Python investigation evaluates selected numeric business rules. It does not independently establish complete categorical domain compliance.

**DQ009 — Referential Integrity**

The Python investigation examines selected customer and channel references to support the assessment of missing or unresolved dimensional relationships.

The authoritative DQ009 findings, exception population, and control status remain documented in the master workbook.

### METHODOLOGY

1. Identify selected missing or unresolved references.
2. Evaluate applicable numeric values against documented business rules.
3. Quantify affected records within the tested scope.
4. Produce supporting investigation evidence.
5. Reconcile relevant findings with the master Data Quality workbook.

### GOVERNANCE

- Source records remain unchanged.
- Python findings supplement, rather than replace, the complete Data Quality audit.
- Statistical anomalies and confirmed business-rule violations are treated separately.
- A detected violation does not automatically authorize remediation.
- Any unresolved business interpretation must follow the approved clarification process.
- The master workbook remains the authoritative record for control status, evidence, and follow-up decisions.

---



In [14]:

# ============================================================
# BR001 | AUTOMATED DATA QUALITY ASSESSMENT
# DQ004 / DQ007 / DQ009 — SUPPORTING INVESTIGATION
# ============================================================

# OBJECTIVE:
# Investigate selected missing references and numeric
# business-rule violations in the BR001 analytical extract.
#
# GOVERNANCE:
# - Read-only assessment.
# - Does not replace the full master workbook audit.
# - No automatic remediation.
# ============================================================

# ------------------------------------------------------------
# 02.1 | VALIDATION RULES
# ------------------------------------------------------------

required_references = [
    "customer_id",
    "channel_id"
]

numeric_rules = {
    "quantity": lambda s: s <= 0,
    "unit_price": lambda s: s <= 0,
    "unit_cost": lambda s: s < 0,
    "discount_pct": lambda s: (s < 0) | (s > 1)
}

required_columns = (
    required_references + list(numeric_rules)
)

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )


# ------------------------------------------------------------
# 02.2 | MISSING REFERENCE CHECKS
# ------------------------------------------------------------

dq_results = []

for column in required_references:

    series = df[column]

    # Detect nulls, empty strings, and whitespace-only values
    missing_mask = (
        series.isna() |
        series.astype("string").str.strip().eq("")
    ).fillna(False)

    invalid_count = int(missing_mask.sum())

    dq_results.append({
        "Check": f"Missing {column}",
        "Affected Rows": invalid_count,
        "Status": (
            "FAIL" if invalid_count > 0 else "PASS"
        )
    })


# ------------------------------------------------------------
# 02.3 | NUMERIC BUSINESS-RULE CHECKS
# ------------------------------------------------------------

for column, rule in numeric_rules.items():

    series = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    # Range violations only.
    # Missing or nonparseable values require separate DQ checks.
    invalid_mask = rule(series).fillna(False)

    invalid_count = int(invalid_mask.sum())

    dq_results.append({
        "Check": f"Invalid {column}",
        "Affected Rows": invalid_count,
        "Status": (
            "FAIL" if invalid_count > 0 else "PASS"
        )
    })


# ------------------------------------------------------------
# 02.4 | INVESTIGATION SUMMARY
# ------------------------------------------------------------

dq_summary = pd.DataFrame(dq_results)

display(dq_summary)


# ------------------------------------------------------------
# 02.5 | GOVERNANCE NOTE
# ------------------------------------------------------------

print(
    "Scope: selected reference completeness and "
    "numeric range validations."
)

print(
    "Full DQ004, DQ007, and DQ009 assessments "
    "remain documented in the master workbook."
)

print(
    "No source records were modified."
)


,Check,Affected Rows,Status
0,Missing customer_id,1,FAIL
1,Missing channel_id,1,FAIL
2,Invalid quantity,1,FAIL
3,Invalid unit_price,0,PASS
4,Invalid unit_cost,1,FAIL
5,Invalid discount_pct,1,FAIL


Scope: selected reference completeness and numeric range validations.
Full DQ004, DQ007, and DQ009 assessments remain documented in the master workbook.
No source records were modified.



---

## 03 | Order Identity & Consistency

**Objective:** Investigate whether order identifiers are consistently associated with customer and transaction-date attributes.

**Traceability:** CL-001  
**Supporting Evidence:** `BR001_DQ_Order_Integrity.sql`  
**Master Record:** `Nova_BR001_Data_Quality_Workbook.xlsx`

### METHODOLOGY

1. Identify the transaction-date field.
2. Group sales lines by `order_id`.
3. Calculate distinct customers and dates per order.
4. Quantify conflicting order identifiers.
5. Preserve the results for business clarification.

### GOVERNANCE

- Do not assume `order_id` is globally unique.
- Conflicting attributes require business-key clarification.
- No composite key is approved through this investigation.


In [4]:

# ============================================================
# BR001 | ORDER IDENTITY & CONSISTENCY
# ============================================================

# Identify the transaction date column
date_candidates = [
    "order_date",
    "transaction_date",
    "full_date",
    "date"
]

date_column = next(
    (col for col in date_candidates if col in df.columns),
    None
)

if date_column is None:
    raise KeyError("No transaction date column found.")

# Profile order-level consistency
order_profile = (
    df.groupby("order_id", dropna=True)
      .agg(
          sales_lines=("sales_line_id", "size"),
          distinct_customers=("customer_id", "nunique"),
          distinct_dates=(date_column, "nunique")
      )
      .reset_index()
)

# Identify conflicting order identifiers
order_profile["customer_conflict"] = (
    order_profile["distinct_customers"] > 1
)

order_profile["date_conflict"] = (
    order_profile["distinct_dates"] > 1
)

order_profile["any_conflict"] = (
    order_profile["customer_conflict"] |
    order_profile["date_conflict"]
)

order_consistency_summary = pd.DataFrame({
    "Metric": [
        "Total order identifiers",
        "Multiple customers",
        "Multiple dates",
        "Any conflict",
        "Both conflicts"
    ],
    "Count": [
        len(order_profile),
        order_profile["customer_conflict"].sum(),
        order_profile["date_conflict"].sum(),
        order_profile["any_conflict"].sum(),
        (
            order_profile["customer_conflict"] &
            order_profile["date_conflict"]
        ).sum()
    ]
})

display(order_consistency_summary)


,Metric,Count
0,Total order identifiers,14803
1,Multiple customers,5518
2,Multiple dates,5498
3,Any conflict,5518
4,Both conflicts,5498



---

## 04 | Transaction Status & Financial Exposure

**Objective:** Quantify recorded financial amounts by transaction status to support clarification of partially returned transactions.

**Traceability:** DQ015 / CL-002  
**Source:** BR001 Analytical Extract  
**Master Record:** `Nova_BR001_Data_Quality_Workbook.xlsx`

### METHODOLOGY

1. Group transactions by `order_status`.
2. Count recorded sales lines.
3. Aggregate financial measures.
4. Identify the financial exposure associated with each status.

### GOVERNANCE

- Financial measures represent recorded source amounts.
- No transactions are excluded or adjusted.
- KPI eligibility remains subject to business clarification.


In [5]:

# ============================================================
# BR001 | TRANSACTION STATUS & FINANCIAL EXPOSURE
# ============================================================

financial_columns = [
    "gross_revenue",
    "discount_amount",
    "net_revenue",
    "total_cost",
    "gross_profit"
]

# Financial exposure by transaction status
status_exposure = (
    df.groupby("order_status", dropna=False)
      .agg(
          sales_lines=("sales_line_id", "size"),
          **{
              column: (column, "sum")
              for column in financial_columns
          }
      )
      .reset_index()
)

display(status_exposure)


,order_status,sales_lines,gross_revenue,discount_amount,net_revenue,total_cost,gross_profit
0,Cancelled,767,575504.55,86744.51,488760.04,339203.15,149556.89
1,Completed,21345,16068821.71,2505077.71,13563744.00,9483372.33,4080371.67
2,Partially Returned,1075,791370.43,117863.93,673506.50,470229.86,203276.64
3,Pending,343,247533.98,36007.70,211526.28,149356.31,62169.97



---

## 05 | Statistical Outlier Reconciliation

**Objective:** Reproduce the IQR-based statistical assessment and reconcile detected outlier counts with the Power Query baseline.

**Traceability:** DQ013  
**Method:** Interquartile Range (1.5 × IQR)  
**Supporting Notebook:** `BR001_DQ013_Outlier_Review.ipynb`  
**Master Record:** `Nova_BR001_Data_Quality_Workbook.xlsx`

### METHODOLOGY

1. Calculate Q1 and Q3 for each numeric variable.
2. Calculate the interquartile range.
3. Determine lower and upper outlier boundaries.
4. Count observations outside those boundaries.
5. Reconcile Python counts against Power Query.

### GOVERNANCE

- Statistical outliers are investigation candidates.
- Outlier detection does not establish data invalidity.
- Business validation is required before remediation.


In [15]:
# ------------------------------------------------------------
# 05.1 | POWER QUERY REFERENCE BASELINE
# ------------------------------------------------------------

pq_outliers = {
    "quantity": 997,
    "unit_price": 0,
    "unit_cost": 0,
    "discount_pct": 1,
    "gross_revenue": 2373,
    "discount_amount": 2059,
    "net_revenue": 2328,
    "total_cost": 2611,
    "gross_profit": 2054
}


# ------------------------------------------------------------
# 05.2 | SOURCE AND COLUMN VALIDATION
# ------------------------------------------------------------

expected_rows = 23530

if len(df) != expected_rows:
    raise ValueError(
        f"Unexpected population: {len(df):,} rows. "
        f"Expected: {expected_rows:,}."
    )

missing_columns = [
    column
    for column in pq_outliers
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )


# ------------------------------------------------------------
# 05.3 | IQR OUTLIER CALCULATION
# ------------------------------------------------------------

outlier_results = []

for column, expected_count in pq_outliers.items():

    series = pd.to_numeric(
        df[column],
        errors="raise"
    )

    # Quartile calculation
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    # Interquartile range
    iqr = q3 - q1

    # Statistical boundaries
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    # Identify statistical outliers
    outlier_mask = (
        (series < lower_bound) |
        (series > upper_bound)
    )

    actual_count = int(outlier_mask.sum())

    # Compare Python with Power Query
    difference = actual_count - expected_count

    outlier_results.append({
        "Variable": column,
        "Power Query": expected_count,
        "Python": actual_count,
        "Difference": difference,
        "Status": (
            "PASS"
            if difference == 0
            else "REVIEW"
        )
    })


# ------------------------------------------------------------
# 05.4 | RECONCILIATION RESULTS
# ------------------------------------------------------------

outlier_reconciliation = pd.DataFrame(
    outlier_results
)

display(outlier_reconciliation)


# ------------------------------------------------------------
# 05.5 | RECONCILIATION SUMMARY
# ------------------------------------------------------------

total_variables = len(outlier_reconciliation)

passed = int(
    outlier_reconciliation["Status"]
    .eq("PASS")
    .sum()
)

review = int(
    outlier_reconciliation["Status"]
    .eq("REVIEW")
    .sum()
)

print("=" * 60)
print("BR001 | DQ013 — IQR RECONCILIATION SUMMARY")
print("=" * 60)

print(f"Population: {len(df):,} sales lines")
print(f"Variables assessed: {total_variables}")
print(f"PASS: {passed}")
print(f"REVIEW: {review}")

if review == 0:
    print(
        "\nTECHNICAL RECONCILIATION: PASS"
    )
    print(
        "All Python outlier counts match the "
        "documented Power Query baseline."
    )
else:
    print(
        "\nTECHNICAL RECONCILIATION: REVIEW"
    )
    print(
        "One or more variables do not reconcile. "
        "Investigate before closing TR-001."
    )


# ------------------------------------------------------------
# 05.6 | CONTROL GOVERNANCE
# ------------------------------------------------------------

print("\nDQ013 CONTROL STATUS:")
print("REVIEW — Pending Business Clarification")

print(
    "\nNote: Statistical reconciliation does not "
    "establish business validity or authorize remediation."
)


,Variable,Power Query,Python,Difference,Status
0,quantity,997,997,0,PASS
1,unit_price,0,0,0,PASS
2,unit_cost,0,0,0,PASS
3,discount_pct,1,1,0,PASS
4,gross_revenue,2373,2373,0,PASS
5,discount_amount,2059,2059,0,PASS
6,net_revenue,2328,2328,0,PASS
7,total_cost,2611,2611,0,PASS
8,gross_profit,2054,2054,0,PASS


BR001 | DQ013 — IQR RECONCILIATION SUMMARY
Population: 23,530 sales lines
Variables assessed: 9
PASS: 9
REVIEW: 0

TECHNICAL RECONCILIATION: PASS
All Python outlier counts match the documented Power Query baseline.

DQ013 CONTROL STATUS:
REVIEW — Pending Business Clarification

Note: Statistical reconciliation does not establish business validity or authorize remediation.



## 06 | DQ013 — Investigation Findings

**Control:** DQ013 — Statistical Outlier Assessment  
**Control Status:** REVIEW — Pending Business Clarification  
**Related Clarification:** CL-007 — OPEN  
**Supporting Notebook:** `BR001_DQ013_Outlier_Review.ipynb`  
**Master Record:** `Nova_BR001_Data_Quality_Workbook.xlsx`

### Detection Reconciliation

**Technical Result:** PASS — subject to successful execution of the nine-variable reconciliation.

All nine IQR outlier counts must reconcile between Python and the documented Power Query baseline.

This technical result does not represent final business approval of DQ013.

### Investigation Findings

- Quantity outliers include extreme transaction quantities.
- High-cost observations require commercial context.
- Negative gross-profit cases were identified within the high-cost population.
- Statistical extremity alone does not establish a data-quality defect.

### Limitations

The investigation identifies unusual observations and financial exposure but cannot independently confirm the business legitimacy of each transaction.

Statistical findings must not be interpreted as confirmed data-quality defects without supporting business evidence.

### Clarification Requirements

**CL-007 — OPEN**

Business stakeholders must validate the treatment of unusual quantities, high-cost transactions, and negative-profit cases where applicable.

The clarification must establish whether the observations represent legitimate commercial activity, source-data errors, or transactions requiring further investigation.

### Governance

- Preserve the original source records.
- Do not automatically remove or modify statistical outliers.
- Do not authorize remediation based solely on IQR results.
- Record business decisions in the master workbook.
- Revalidate affected controls after any approved treatment.

### Final Assessment

**DQ013 Status:** REVIEW — Pending Business Clarification.

**Next Step:** Business clarification through CL-007 before determining final treatment or KPI eligibility.



---

## 07 | Duplicate Sales Line Investigation

**Objective:** Investigate repeated `sales_line_id` values and compare the corresponding transaction records.

**Traceability:** DQ005 / CL-004  
**Supporting Evidence:** `BR001_DQ_Order_Integrity.sql`  
**Master Record:** `Nova_BR001_Data_Quality_Workbook.xlsx`

### METHODOLOGY

1. Identify all records sharing a sales-line identifier.
2. Compare records across available source attributes.
3. Identify conflicting record versions.
4. Quantify affected populations and recorded financial exposure.

### GOVERNANCE

- Duplicate identifiers do not automatically imply identical transactions.
- No records are deleted.
- Remediation requires documented classification and approval.


In [7]:

# ============================================================
# BR001 | DUPLICATE SALES LINE INVESTIGATION
# ============================================================

# Identify all records sharing a sales line identifier
duplicates = df[
    df["sales_line_id"].duplicated(keep=False)
].copy()

# Compare record versions across all source columns
duplicate_profile = (
    duplicates.groupby("sales_line_id")
    .agg(
        occurrences=("sales_line_id", "size"),
        distinct_orders=("order_id", "nunique"),
        distinct_customers=("customer_id", "nunique"),
        distinct_dates=("order_date", "nunique"),
        recorded_net_revenue=("net_revenue", "sum"),
        recorded_gross_profit=("gross_profit", "sum")
    )
    .reset_index()
)

display(duplicate_profile)


,sales_line_id,occurrences,distinct_orders,distinct_customers,distinct_dates,recorded_net_revenue,recorded_gross_profit
0,SL0000251,2,2,2,2,284.75,110.16
1,SL0008001,2,2,2,2,2688.50,1341.90
2,SL0017001,2,2,2,2,1804.97,771.70


In [8]:

# ============================================================
# BR001 | DUPLICATE RECORD DETAIL COMPARISON
# ============================================================

# Compare duplicate record versions
comparison_columns = [
    "sales_line_id",
    "order_id",
    "order_date",
    "customer_id",
    "channel_name",
    "quantity",
    "unit_price",
    "net_revenue",
    "gross_profit"
]

duplicate_comparison = (
    duplicates[comparison_columns]
    .sort_values(
        by=["sales_line_id", "order_id"]
    )
)

display(duplicate_comparison)


,sales_line_id,order_id,order_date,customer_id,channel_name,quantity,unit_price,net_revenue,gross_profit
1,SL0000251,O0000169,2025-01-01,C01129,E-commerce,1,189.69,180.21,74.79
2191,SL0000251,O0000170,2025-02-05,C01471,Retail Store,1,130.67,104.54,35.37
5350,SL0008001,O0005090,2025-03-27,C02329,Marketplace,1,186.03,148.82,4.35
8217,SL0008001,O0005091,2025-05-13,C01997,Retail Store,3,846.56,2539.68,1337.55
20654,SL0017001,O0010657,2026-05-19,C01498,Marketplace,1,823.51,782.33,370.44
22836,SL0017001,O0010658,2026-06-20,C01325,Telephone Sales,2,511.32,1022.64,401.26



---

## 08 | Invalid Numeric Values

**Objective:** Quantify numeric business-rule violations and their recorded financial exposure.

**Traceability:** DQ007  
**Source:** BR001 Analytical Extract  
**Master Record:** `Nova_BR001_Data_Quality_Workbook.xlsx`

### BUSINESS RULES

- Quantity must be greater than zero.
- Unit cost must not be negative.
- Discount percentage must remain within the permitted range.

### METHODOLOGY

1. Apply numeric validation rules.
2. Count affected records by rule.
3. Quantify associated financial measures.
4. Preserve results for the master DQ register.

### GOVERNANCE

- Recorded financial amounts are not automatically losses.
- Violations require investigation before correction.
- Original source records remain unchanged.


In [10]:

# ============================================================
# BR001 | INVALID NUMERIC VALUES
# ============================================================

# Confirmed numeric business rule violations
invalid_rules = {
    "quantity": df["quantity"] <= 0,
    "unit_cost": df["unit_cost"] < 0,
    "discount_pct": (
        (df["discount_pct"] < 0) |
        (df["discount_pct"] > 1)
    )
}

invalid_results = []

for rule_name, mask in invalid_rules.items():
    affected = df.loc[mask]

    invalid_results.append({
        "Rule": rule_name,
        "Affected Rows": len(affected),
        "Recorded Net Revenue": round(
            affected["net_revenue"].sum(), 2
        ),
        "Recorded Gross Profit": round(
            affected["gross_profit"].sum(), 2
        )
    })

invalid_numeric_summary = pd.DataFrame(invalid_results)

display(invalid_numeric_summary)


,Rule,Affected Rows,Recorded Net Revenue,Recorded Gross Profit
0,quantity,1,-1005.40,-344.78
1,unit_cost,1,564.26,569.01
2,discount_pct,1,-95.71,-494.87
